# Logistic Regression
This is an example of performing logistic regression in Python with the scikit-learn module. In this example, we perform many useful python functions beyond what we need for a simple model. This is done partially to explore some more advanced modeling, array maniputation, evaluation, and so on. We will also use several external modules importing data, formatiing data, manipulating data,modeling and graphical exploration.


# Dataset
The dataset I chose for this example in Longitudinal Low Birth Weight Study (CLSLOWBWT.DAT). Hosmer and Lemeshow (2000) Applied Logistic Regression:
 Second Edition.  These data are copyrighted by John Wiley &
 Sons Inc. and must be acknowledged and used accordingly.




# List of Variables:

Variable	Description		Codes/Values		Name
______________________________________________________________________

1	Identification Code		ID Number		ID

2	Birth Number			1-4			BIRTH

3	Smoking Status 			0 = No, 1 = Yes		SMOKE
	During Pregnancy

4	Race				1 = White, 2 = Black 	RACE
					3  = Other

5	Age of Mother			Years			AGE

6	Weight of Mother at 		Pounds			LWT
	Last Menstrual Period

7	Birth Weight			Grams			BWT

8	Low Birth Weight		1 = BWT <=2500g,	LOW
					0 = BWT >2500g
________________________________________________________________________

# Problem Statement
In this example, we want to predict Low Birth Weight using the reamining dateset variables. Low Birth Weight, the dependent variable, 1 = BWT <=2500g and 0 = BWT >2500g.

# Import Modules

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from patsy import dmatrices
from sklearn.linear_model import LogisticRegression
from sklearn.cross_validation import train_test_split
from sklearn import metrics
from sklearn.cross_validation import cross_val_score

# Data Pre-Processing 

In [ ]:
# load dataset
dta = pd.read_csv("C:\Users\Strickland\Documents\Python Scripts\CLSLOWBWT.csv")

# Data Exploration

In [ ]:
dta.groupby('LOW').mean()

We can see that on average, women who have children with a low birth weight are more likely to be smokers than nonsmokers. This makes sense based on scientific studies. Let's take another look at the Birth Number variable.

In [ ]:
dta.groupby('BIRTH').mean()

Low Birth Weight trends upward with more births.

# Data Visualization

In [ ]:
# show plots in the notebook
%matplotlib inline

In [ ]:
# histogram of birth number
dta.BIRTH.hist()
plt.title('Histogram of Low Birth Weight')
plt.xlabel('Birth Number')
plt.ylabel('Frequency')


In [ ]:
# histogram of age of mother
dta.AGE.hist()
plt.title('Histogram of Age of Mother')
plt.xlabel('Age')
plt.ylabel('Frequency')

Let's take a look at the distribution of smokers for those having children with low birth weights versus those who do not.

In [ ]:
# barplot of low birth weights grouped by smoker status (True or False)
pd.crosstab(dta.SMOKE, dta.LOW.astype(bool)).plot(kind='bar')
plt.title('Somker Distribution by Low Birth Weight')
plt.xlabel('Smoker')
plt.ylabel('Frequency')


Now let's use a stacked barplot to look at the percentage of women having children with low birth weights by age.

In [ ]:
low_age = pd.crosstab(dta.AGE, dta.LOW.astype(bool))
low_age.div(low_age.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Low Birth Weight Percentage by Age of Mother')
plt.xlabel('Age of Mother')
plt.ylabel('Percentage')


# Prepare Data for Logistic Regression?
To prepare the data, we want to add an intercept column as well as dummy variables for Age of Mother and Weight of Mother at Last Menstrual Period, since we are treating them as categorial variables. The dmatrices function from the patsy module can do that using formula language.

The column names for the dummy variables are messy, so let's rename those.

In [ ]:
# create dataframes with an intercept column and dummy variables for
y, X = dmatrices('LOW ~ ID + BIRTH + SMOKE + RACE + AGE + LWT',
                  dta, return_type="dataframe")
print X.columns

We now want to convert the numeric (interval) variable AGE to a categorical variable with 4 classes.

We also need to flatten y into a 1-D array, so that scikit-learn will properly understand it as the response variable.

In [ ]:
# flatten y into a 1-D array
y = np.ravel(y)

#First Logistic Regression?
Let's go ahead and run logistic regression on the entire data set, and see how accurate it is.


In [ ]:
# instantiate a logistic regression model, and fit with X and y
model = LogisticRegression()
model = model.fit(X, y)

# check the accuracy on the training set
model.score(X, y)

70.7% accuracy seems good, but what's the null error rate?

In [ ]:
# what percentage had low birth weights?
y.mean()

Only 31% of the women had low birth rate children, which means that you could obtain 69% accuracy by always predicting "no". So we're doing better than the null error rate, but not by much.
Let's examine the coefficients to see what we learn.

In [ ]:
# examine the coefficients
pd.DataFrame(zip(X.columns, np.transpose(model.coef_)))

# Coverting Variables & Merging Dataframes
Now, we want to take the varible AGE and convert it to a categorical variable, to see if we can improve the model. We will do this by creating two data frames deom our original data. we will then merge the two data frames, so both need to contain the ID variable. One data from will have the converted categorical variable age_group, and the other will have the dependent variable and the other indepedent variables.

In [ ]:
df1 = pd.DataFrame(dta, columns=['ID','AGE'])
df2 = pd.DataFrame(dta, columns=['ID', 'BIRTH', 'SMOKE', 'RACE', 'LWT', 'LOW'])
bins = [15, 25, 35, 45, 55]
group_names = ['15-24', '25-34', '35-44', '45-55']
age_groups = pd.cut(df1['AGE'], bins, labels=group_names)
df1['age_groups'] = pd.cut(df1['AGE'], bins, labels=group_names)
categories
df1.head(5)

We now merge the two data frames.

In [ ]:
left = df2
right = df1
result = pd.merge(left, right, on='ID')
result.head(5)

# Second Logistic Regression
We are now ready to build and evaluate our second logistic regression model, youing the merged data frames. 

In [ ]:
y, Z = dmatrices('LOW ~ BIRTH + SMOKE + RACE + age_groups + LWT', result, return_type="dataframe")
print Z.columns

Since we change the size of y when we converted AGE and merged, we also need to flatten y again into a 1-D array, so that scikit-learn will properly understand it as the response variable

In [ ]:
# flatten y into a 1-D array
y = np.ravel(y)

Before we perform the logistic regression, we want to check the matrix we just formed to ensure it is consistent with our intent.

In [ ]:
Z.head(5)

Finaly, we are ready to execute the logistic regresson model and see how accurate it is.

In [ ]:
# instantiate a logistic regression model, and fit with X and y
model1 = LogisticRegression()
model1 = model1.fit(Z, y)

# check the accuracy on the training set
model1.score(Z, y)

71.3% accuracy seems good, but what's the null error rate? We also want to recheck the percentage of low birth weights.

In [ ]:
# what percentage had low birth weights?
y.mean()

Still, only 31% of the women had low birth rate children, which means that you could obtain 69% accuracy by always predicting "no". So we're doing better than the null error rate, but not by much. Let's examine the coefficients to see what we learn.

In [ ]:
# examine the coefficients
pd.DataFrame(zip(Z.columns, np.transpose(model.coef_)))

Increases in Birth Number and RACE correspond to a decrease in the likelihood of having an Low Birth Weight child. A decrease in Smoking Status corresponds to a decrease in the likelihood of having an Low Birth Weight child. For Age Group, the lowest likelihood of having Low Birth Weight child corresponds to the baseline age group (15-24), since all of the dummy coefficients are positive.

# Model Evaluation Using a Validation Set?
So far, we have trained and tested on the same set. Let's instead split the data into a training set and a testing set.


In [ ]:
# evaluate the model by splitting into train and test sets
Z_train, Z_test, y_train, y_test = train_test_split(Z, y, test_size=0.3, random_state=0)
model2 = LogisticRegression()
model2.fit(Z_train, y_train)


We now need to predict class labels for the test set. We will also generate the class probabilities, just to take a look.

In [ ]:
# predict class labels for the test set
predicted = model2.predict(Z_test)
print predicted

In [ ]:
# generate class probabilities
probs = model2.predict_proba(Z_test)
print probs

As you can see, the classifier is predicting a 1 (having a Low Birth Weight child) any time the probability in the second column is greater than 0.5.
Now let's generate some evaluation metrics.

In [ ]:
# generate evaluation metrics
print metrics.accuracy_score(y_test, predicted)
print metrics.roc_auc_score(y_test, probs[:, 1])

The accuracy is 67.8%, which is the close to what we experienced when training and predicting on the same data.
We can also see the confusion matrix and a classification report with other metrics.

In [ ]:
print metrics.confusion_matrix(y_test, predicted)
print metrics.classification_report(y_test, predicted)

#Model Evaluation Using Cross-Validation?
Now let's try 10-fold cross-validation, to see if the accuracy holds up more rigorously.

In [ ]:
# evaluate the model using 10-fold cross-validation
scores = cross_val_score(LogisticRegression(), Z, y, scoring='accuracy', cv=10)
print scores
print scores.mean()

Looks good. It's still performing at 69% accuracy.

# Predicting the Probability of Low Birth Weight Child?
Just for fun, let's predict the probability of a low birth weight child for a random woman not present in the dataset. She's a 35-year-old Other race, has had 2 births,(has 2 children), is a smoker,and her weight is 132.

In [ ]:
model.predict_proba(np.array([0, 0, 1, 1, 3, 2, 1]))


The predicted probability of low birth weight child is 39.3%

# Next Steps?
There are many different steps that could be tried in order to improve the model:

? including interaction terms

? removing features

? regularization techniques

? using a non-linear model